# CDLD training for microbial coculture prediction

Train the discovery models and prediction models, select models using validation data, and evaluate performance across training-data fractions. The cells use the model code and data splits in full/ and fractions/.
Run 00_Prepare_Data.ipynb first. Outputs are generated when this source notebook is executed.

In [1]:
from pathlib import Path
import os,sys,json,time,subprocess,concurrent.futures,platform
R=Path.cwd();assert (R/'full/primary/model.py').exists()
assert platform.system()=='Darwin', 'This run is designated macOS reproduction'
SEEDS=[17,43,101,137,211]
def run(args):
 env=os.environ.copy();env.update(OMP_NUM_THREADS='2',OPENBLAS_NUM_THREADS='2',VECLIB_MAXIMUM_THREADS='2')
 tag='_'.join(str(x).replace('/','_') for x in args)[-170:];logs=R/'logs';logs.mkdir(exist_ok=True)
 with (logs/(tag+'.log')).open('a') as f:
  subprocess.run([sys.executable,*map(str,args)],cwd=R,env=env,stdout=f,stderr=subprocess.STDOUT,check=True)
 return tag
def batch(jobs,label):
 print(label, 'jobs:',len(jobs),flush=True)
 with concurrent.futures.ThreadPoolExecutor(max_workers=4) as pool:
  futures=[pool.submit(run,j) for j in jobs]
  for k,f in enumerate(concurrent.futures.as_completed(futures),1):
   tag=f.result();print(label,k,'/',len(jobs),tag,flush=True)
   (R/'status.json').write_text(json.dumps(dict(phase=label,completed=k,total=len(jobs),updated=time.time())))


In [2]:
batch([[R/'full'/sp/'run_job.py','source','gm',s,'source'] for sp in ['primary','robust'] for s in SEEDS], '24h_discovery')

24h_discovery jobs: 10


24h_discovery 1 / 10 full_primary_run_job.py_source_gm_101_source


24h_discovery 2 / 10 full_primary_run_job.py_source_gm_17_source


24h_discovery 3 / 10 full_primary_run_job.py_source_gm_43_source


24h_discovery 4 / 10 full_primary_run_job.py_source_gm_137_source


...

24h_discovery 6 / 10 full_primary_run_job.py_source_gm_211_source


24h_discovery 7 / 10 full_robust_run_job.py_source_gm_17_source


24h_discovery 8 / 10 full_robust_run_job.py_source_gm_101_source


24h_discovery 9 / 10 full_robust_run_job.py_source_gm_137_source


24h_discovery 10 / 10 full_robust_run_job.py_source_gm_211_source


In [3]:
batch([[R/'full'/sp/'run_job.py','reader','gm',s,t] for sp in ['primary','robust'] for t in ['hour48','trio'] for s in SEEDS]+[[R/'full/primary/run_job.py','aug','gm',s,'source'] for s in SEEDS], 'full_predictors')

full_predictors jobs: 25


full_predictors 1 / 25 full_primary_run_job.py_reader_gm_17_hour48


full_predictors 2 / 25 full_primary_run_job.py_reader_gm_101_hour48


full_predictors 3 / 25 full_primary_run_job.py_reader_gm_43_hour48


full_predictors 4 / 25 full_primary_run_job.py_reader_gm_137_hour48


...

full_predictors 21 / 25 full_primary_run_job.py_aug_gm_43_source


full_predictors 22 / 25 full_primary_run_job.py_aug_gm_137_source


full_predictors 23 / 25 full_primary_run_job.py_aug_gm_17_source


full_predictors 24 / 25 full_primary_run_job.py_aug_gm_101_source


full_predictors 25 / 25 full_primary_run_job.py_aug_gm_211_source


In [4]:
batch([[R/'full/baselines.py',R/'full/primary']], 'primary_baselines')
batch([[R/'full/baselines.py',R/'full/robust']], 'robust_baselines')

primary_baselines jobs: 1


primary_baselines 1 / 1 baselines_full_primary


robust_baselines jobs: 1


robust_baselines 1 / 1 baselines_full_robust


In [5]:
items=json.loads((R/'fractions/subsets.json').read_text())
# 100% LF is fitted again to verify the adapter's validation predictions.
batch([[R/'fractions/train.py','q100_r00','LF',s,'reader'] for s in SEEDS], '100pct_gate')
import numpy as np
for s in SEEDS:
 a=np.load(R/f'full/primary/runs/reader_hour48_gm_{s}/validation_predictions.npz')
 b=np.load(R/f'fractions/subsets/q100_r00/LF/runs/reader_hour48_initial_{s}/validation_predictions.npz')
 np.testing.assert_array_equal(a['row_id'],b['row_id']);np.testing.assert_allclose(a['prediction'],b['prediction'],rtol=0,atol=1e-7)
print('100% validation gate passed')

100pct_gate jobs: 5


100pct_gate 1 / 5 fractions_train.py_q100_r00_LF_17_reader


100pct_gate 2 / 5 fractions_train.py_q100_r00_LF_101_reader


100pct_gate 3 / 5 fractions_train.py_q100_r00_LF_43_reader


100pct_gate 4 / 5 fractions_train.py_q100_r00_LF_137_reader


100pct_gate 5 / 5 fractions_train.py_q100_r00_LF_211_reader


100% validation gate passed


In [6]:
batch([[R/'fractions/train.py',x['name'],c,s,'source'] for x in items for c in ['LU','RU'] for s in SEEDS], '48h_discovery')

48h_discovery jobs: 310


48h_discovery 1 / 310 fractions_train.py_q010_r00_LU_101_source


48h_discovery 2 / 310 fractions_train.py_q010_r00_LU_43_source


48h_discovery 3 / 310 fractions_train.py_q010_r00_LU_137_source


48h_discovery 4 / 310 fractions_train.py_q010_r00_LU_17_source


...

48h_discovery 306 / 310 fractions_train.py_q050_r09_RU_43_source


48h_discovery 307 / 310 fractions_train.py_q050_r09_RU_17_source


48h_discovery 308 / 310 fractions_train.py_q050_r09_RU_101_source


48h_discovery 309 / 310 fractions_train.py_q050_r09_RU_137_source


48h_discovery 310 / 310 fractions_train.py_q050_r09_RU_211_source


In [7]:
batch([[R/'fractions/train.py',x['name'],c,s,'reader'] for x in items for c in ['LF','LU','RU'] for s in SEEDS if not(x['pct']==100 and c=='LF')], '48h_predictors')

48h_predictors jobs: 460


48h_predictors 1 / 460 fractions_train.py_q010_r00_LF_43_reader


48h_predictors 2 / 460 fractions_train.py_q010_r00_LF_101_reader


48h_predictors 3 / 460 fractions_train.py_q010_r00_LF_17_reader


48h_predictors 4 / 460 fractions_train.py_q010_r00_LF_137_reader


...

48h_predictors 456 / 460 fractions_train.py_q050_r09_RU_43_reader


48h_predictors 457 / 460 fractions_train.py_q050_r09_RU_17_reader


48h_predictors 458 / 460 fractions_train.py_q050_r09_RU_101_reader


48h_predictors 459 / 460 fractions_train.py_q050_r09_RU_137_reader


48h_predictors 460 / 460 fractions_train.py_q050_r09_RU_211_reader


In [2]:
items=json.loads((R/'fractions/subsets.json').read_text())
batch([[R/'fractions/baselines.py',R/'fractions/subsets'/x['name']] for x in items if x['pct']!=100], 'fraction_baselines')

fraction_baselines jobs: 30


fraction_baselines 1 / 30 baselines_q010_r01


fraction_baselines 2 / 30 baselines_q010_r00


fraction_baselines 3 / 30 baselines_q025_r00


fraction_baselines 4 / 30 baselines_q050_r00


...

fraction_baselines 26 / 30 baselines_q025_r08


fraction_baselines 27 / 30 baselines_q010_r09


fraction_baselines 28 / 30 baselines_q050_r08


fraction_baselines 29 / 30 baselines_q025_r09


fraction_baselines 30 / 30 baselines_q050_r09


In [3]:
run([R/'fractions/evaluate.py'])
import pandas as pd
metrics=pd.read_json(R/'fractions/evaluation/metrics.json')
display(metrics.groupby(['pct','model']).mse.mean().unstack())
display(pd.read_json(R/'fractions/evaluation/contrasts.json'))
(R/'status.json').write_text(json.dumps(dict(phase='complete',updated=time.time())))

model,LF,LU,RU,constant,rf,ridge,xgb
pct,,,,,,,
10,0.072202,0.086420,0.238181,0.679052,0.385685,0.318932,0.344440
25,0.056405,0.059474,0.083960,0.674796,0.264268,0.222653,0.229873
50,0.049490,0.047817,0.054620,0.673173,0.156442,0.191673,0.191764
100,0.046369,0.042817,0.045169,0.672369,0.072455,0.180015,0.178505


,pct,contrast,family,size,difference,interval
0,10,strategy,primary,4,-0.165980,"[-0.19071598902574802, -0.143281806465928]"
1,10,updated_source,secondary,20,-0.151762,"[-0.18074927553851503, -0.126171510390116]"
2,10,learned_update,secondary,20,0.014218,"[0.010548823081750001, 0.018246328089285002]"
3,10,ridge,baseline,12,-0.246731,"[-0.29468665658283805, -0.20268438100840502]"
4,10,rf,baseline,12,-0.313483,"[-0.37649322616164504, -0.25412198982536904]"
5,10,xgb,baseline,12,-0.272238,"[-0.313739308888609, -0.233496675025043]"
6,25,strategy,primary,4,-0.027556,"[-0.033823751548444005, -0.021742717329817002]"
7,25,updated_source,secondary,20,-0.024486,"[-0.031475860945186, -0.01833988123823]"
8,25,learned_update,secondary,20,0.003070,"[0.000310091779325, 0.005953370656867]"
9,25,ridge,baseline,12,-0.166248,"[-0.201535742951742, -0.13551957179513102]"


52